# Transfer to campaign storage, in new ARCO format

Single zarr store per model / methodology, in Zarr 3, as on earthmover.

In [1]:
import xarray as xr
import numpy as np
import pandas as pd
import os
import re
import glob
import zarr
import warnings
from tqdm.notebook import tqdm
from dask.diagnostics import ProgressBar

import distributed

from funcs_support import get_filepaths,get_params,utility_save
from funcs_aux import _find_main_variable, _load_gwls
dir_list = get_params()

In [2]:
from distributed import Client
# Start dask client
client = Client()
display(client)

/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 46565 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc2/proxy/46565/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc2/proxy/46565/status,Workers: 6
Total threads: 18,Total memory: 72.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:39369,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc2/proxy/46565/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:35821,Total threads: 3
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc2/proxy/35223/status,Memory: 12.00 GiB
Nanny: tcp://127.0.0.1:40165,


In [3]:
def load_for_concat(row,
                    chunks = {'lat':10,'lon':10,'gwl':1,'dayofyear':-1,'year':-1},
                    meta_dims = ['exp','run','proj_base'],
                    idv_dims = None):
    '''
    if idv_dims is not None, then only stack over those dims. If None, 
    then the meta_dims are used. 
    '''
    # Load
    ds = xr.open_zarr(row[1]['path'],consolidated=False)

    # Rechunk
    if chunks is not None:
        ds = ds.chunk(**{dim:chunk for dim,chunk in chunks.items() if dim in ds})

    # Add metadata
    ds = ds.assign_coords({v:[row[1][v]] for v in meta_dims})
    if 'exp' in meta_dims:
        ds = ds.rename({'exp':'experiment'})
        meta_dims = ['experiment' if k == 'exp' else k for k in meta_dims]

    if 'binF' in row[1]['varname']:
        ds = ds.rename({'bin':'binF','bin_bnds':'binF_bnds'})
        ds = ds.set_coords('binF_bnds')
    elif 'binC' in row[1]['varname']:
        ds = ds.rename({'bin':'binC','bin_bnds':'binC_bnds'})
        ds = ds.set_coords('binC_bnds')

    # Stack the metadata to a single variable 
    if idv_dims is None:
        idv_dims = meta_dims
    ds = ds.stack(idv = idv_dims)

    return ds

In [4]:
output_dir = dir_list['bcd_me']

In [10]:
#var = 'tas' # or tasmax
flists = {'qdm':get_filepaths(source_dir = 'proc').query(f'(varname == "tas" or varname == "tasmax") and proj_method == "QDM" and dwnscl_method != dwnscl_method'),
          'qdm-qplad':(get_filepaths(source_dir = 'proc').query(f'proj_method == "QDM" and dwnscl_method == "QPLAD"'))}
                                                         #loc[lambda d: d['varname'].str.contains(r'^'+var+r'(?!max)')])}

# Metadata, for a general dataset DESCRIPTION attribute (specific model and proj_base info is added to this)
descs = {'qdm':'ScenarioMIP runs, bias-corrected using Quantile Delta Mapping (QDM, Cannon et al. 2015)',
         'qdm-qplad':'statistics of ScenarioMIP runs, bias-corrected using Quantile Mapping (QDM, Cannon et al. 2015) and downscaled using Quantile-Preserving Localized Analog-Downscaling (QPLAD, Gergel et al. 2024)'}


In [11]:
# Get GWL info
gwl_info = _load_gwls()

In [12]:
meth = 'qdm'
flist = flists['qdm']
modlist = ['ACCESS-ESM1-5', 'CanESM5', 'EC-Earth3', 'EC-Earth3-Veg',
               'IPSL-CM6A-LR', 'MIROC6', 'MPI-ESM1-2-LR']
mod = 'EC-Earth3-Veg'
output_fn = f'{output_dir}bcd_me_{meth}_{mod}.zarr'

print(f'Processing {meth}, {mod}!')

if 'qplad' in meth:
    chunks={'lat':200,'lon':250,'gwl':1}
else:
    chunks = {'lat':40,'lon':40,'gwl':1,'dayofyear':-1,'year':-1}

Processing qdm, EC-Earth3-Veg!


In [ ]:
for meth,flist in flists.items():
    #modlist = np.unique(flist.model.values)
    modlist = ['ACCESS-ESM1-5', 'CanESM5', 'EC-Earth3', 'EC-Earth3-Veg',
               'IPSL-CM6A-LR', 'MIROC6', 'MPI-ESM1-2-LR']
    for mod in modlist:
    
        output_fn = f'{output_dir}bcd_me_{meth}_{mod}.zarr'

        if not os.path.exists(output_fn+'/.done_v021'):
        
            print(f'Processing {meth}, {mod}!')

            if 'qplad' in meth:
                chunks={'lat':200,'lon':250,'gwl':1}
            else:
                chunks = {'lat':40,'lon':40,'gwl':1,'dayofyear':-1,'year':-1}

            # Load into single dask store
            with warnings.catch_warnings():
                warnings.filterwarnings('ignore')
                # Turning off increasing chunk number warning
                dss = xr.combine_by_coords([load_for_concat(row,idv_dims = ['experiment','run'],chunks=chunks) for row in flist.query(f'model == "{mod}"').iterrows()],
                                data_vars = 'all', # Prevents of the xarray depreciation errors, though unclear what causes it. Output looks as it should... 
                                join='outer', # To get all GWLs
                                           combine_attrs = 'drop'
                               )
                dss = dss.chunk(chunks) # Doesn't always keep for all variables
            
            # Dayofyear is float for some reason
            if 'dayofyear' in dss:
                dss['dayofyear'] = dss.dayofyear.astype(int)
            
            #---- Metacoordinates
            # Add in calendar years within the model of the GWL
            gwl_info_tmp = (gwl_info.loc[mod].reset_index().
                        rename({'warming_level':'gwl',
                                'exp':'experiment',
                                'ensemble':'run'},axis=1).
                        set_index(['experiment','run','gwl']))
            gwl_info_tmp = gwl_info_tmp.sort_index()
            
            calendar_years = xr.DataArray(dims = ['idv','gwl','year'],
                         coords = {'idv':(['idv'],dss.idv.values),
                                   'gwl':(['gwl'],dss.gwl.values),
                                   'year':(['year'],np.arange(1,21))})
            for idx in calendar_years.idv.values:
                calendar_years.loc[{'idv':idx,
                                'gwl':gwl_info_tmp.loc[idx[0:2]].index}] = [np.arange(s,e) for s,e in zip(gwl_info_tmp.loc[idx[0:2]]['start_year'],
                                                                                                          gwl_info_tmp.loc[idx[0:2]]['end_year']+1)]

            # Reset index for export (no multiindices supported)
            dss = dss.reset_index('idv')
            # Assign calendar year coords, but with reset index (the order is correctly 
            # set before the resetting)
            dss = dss.assign_coords(calendar_year = calendar_years.reset_index('idv'))

            # Drop idv, since multi-indices aren't supported for saving
            dss = dss.drop_vars('idv')
            
            # Take out the ISIMIP 0.5-deg ERA5, if present
            dss = dss.sel(proj_base = [p for p in dss.proj_base.values if p != 'ERA5'])
            
            # Rename the "ERA5-025" (originally in contrast to ISIMIP ERA5, 
            # which was falsely named as just "ERA5")
            dss['proj_base'] = [re.sub(r'ERA5\-025','ERA5',m) for m in dss.proj_base.values]

            # Add a metadata coordinate for which runs / GWLs have data for a given variable
            has_data = ((~np.isnan(dss.isel(proj_base=0,drop=True).
                                  sel(lat=20,lon=-100,method='nearest',drop=True))).
                        any([d for d in dss.dims if d not in ['idv','proj_base','gwl','lat','lon']])).compute()
            dss = dss.assign_coords({'has_data':has_data.to_dataarray(name='has_data')})

            # Add 'model' coordinate to make concatenating across models easier 
            dss = dss.assign_coords({'model':(('idv'),[mod]*dss.sizes['idv'])})
            
            #---- Attribute metadata
            # Set variable-level metadata
            if np.any([re.search('tas[(bin)|(sum)|(mx)]',var) for var in dss]):
                basevar = 'tas'
                vardesc = 'Mean Near-Surface Air Temperature'
            elif np.any([re.search('tasmax[(bin)|(sum)|(mx)]',var) for var in dss]):
                basevar = 'tasmax'
                vardesc = 'Maximum Near-Surface Air Temperature'
            
            if 'tas' in dss:
                dss['tas'].attrs = {'long_name':'Daily Near-Surface Air Temperature',
                             'standard_name':'air_temperature',
                             'units':'K'}
            if 'tasmax' in dss:
                dss['tasmax'].attrs = {'long_name':'Daily Maximum Near-Surface Air Temperature',
                                 'standard_name':'max_air_temperature',
                                 'units':'K'}
            if basevar+'binC' in dss:
                dss[basevar+'binC'].attrs =  {'long_name':f'Days With {vardesc} in Celsius Bins',
                                         'standard_name':'C_bin_days',
                                         'units':'days'}
                # Make sure ocean bits are nans instead of 0 (comes from default 
                # sum as nansum behavior somewhere down the line)
                dss[basevar+'binC'] = dss[basevar+'binC'].where(dss[basevar+'binC'].sum('binC') > 0)
            if basevar+'binF' in dss:
                dss[basevar+'binF'].attrs =  {'long_name':f'Days With {vardesc} in Fahrenheit Bins',
                                             'standard_name':'F_bin_days',
                                             'units':'days'}
                # Make sure ocean bits are nans instead of 0 (comes from default 
                # sum as nansum behavior somewhere down the line)
                dss[basevar+'binF'] = dss[basevar+'binF'].where(dss[basevar+'binF'].sum('binF') > 0)
            if basevar+'sumpoly' in dss:
                dss[basevar+'sumpoly'].attrs =  {'long_name':f'Polynomial Sums of {vardesc} Per Year',
                                             'standard_name':basevar+'_sum_poly',
                                             'units':'C^k'}
            
            # Set coordinate-level metadata
            if 'binF' in dss:
                dss.binF.attrs = {'long_name':'Fahrenheit Bins','short_name':'bins'}
                dss.binF_bnds.attrs = {'long_name':'Fahrenheit Bin Bounds','short_name':'bin_bnds'}
                dss = dss.chunk({'binF':-1})
            if 'binC' in dss:
                dss.binC.attrs = {'long_name':'Celsius Bins','short_name':'bins'}
                dss.binC_bnds.attrs = {'long_name':'Celsius Bin Bounds','short_name':'bin_bnds'}
                dss = dss.chunk({'binC':-1})
            if 'sumpoly' in dss:
                dss = dss.chunk({'sumpoly':-1})
            
            dss.calendar_year.attrs = {'long_name':'Calendar Year',
                                    'description':"Year within a model run's calendar"}
            dss.experiment.attrs = {'long_name':'Experiment',
                                    'description':'ScenarioMIP Experiment'}
            dss.run.attrs = {'long_name':'Ensemble Member',
                             'description':'Ensemble member ID'}
            dss.proj_base.attrs = {'long_name':'Bias-correction base reanalysis',
                                   'description':'Reanalysis product used as "ground truth" for bias-correction'}
            dss.gwl.attrs = {'long_name':'Global Warming Level',
                             'short_name':'GWL',
                             'description':"Global mean surface temperature of this 20-year segment as anomalies above model run's 1850-1900 average"}
            
            # Set file metadata
            dss.attrs = {
                'DESCRIPTION':f'{mod} {descs[meth]} to {', '.join(np.unique([dss.proj_base.values]))}',
                'Project':'Bias-Corrected, Downscaled Massive Ensemble (BCD-ME)',
                'Creators':'Kevin Schwarzwald, Nathan Lenssen, Radley Horton, and Gernot Wagner',
                'Version':'v0.2.1',
                'License':'CC BY 4.0'
                    }
    
            #---- Write!
            #with ProgressBar():
            utility_save(dss.drop_encoding(),
                         output_fn,zarr_mode = 'w',
                         save_kwargs = {'zarr_format':3})
            # Add v02 done flag
            open(output_fn+'/.done_v021', 'w').close()
